# Model V3 - 15-Minute Price Prediction (V2.5 + Cross-Border Grid)

V3 extends V2.5 with **Finland cross-border grid transmission** features (`fi_*`).
The feature table `V3_15min_features.csv` already contains ALL previous features
(weather, calendar, price lags, price rolling) **plus** the 13 grid columns.

This notebook trains the V3 XGBoost model and compares it against the recorded
V2.5 metrics (MAE 2.82 / RMSE 8.22 / R² 0.972) — the chronological 80/20 split
and hyperparameters are identical, so it is a fair comparison.

In [1]:
import pandas as pd

In [2]:
# load the V3 feature CSV (V2.5 features + grid transmission features)
df = pd.read_csv('../data/convertData/V3_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
df

,datetime,price,temp,wind_speed,wind_direction_deg,wind_dir_sin,wind_dir_cos,hour,minute,day_of_week,...,fi_se_central,fi_se_total,fi_total_net,fi_se_abs,fi_total_net_lag_96,fi_total_net_lag_672,fi_se_total_lag_96,fi_se_total_lag_672,fi_ee_lag_96,fi_ee_lag_672
0,2023-01-01 00:00:00+02:00,4.8400,4.45,8.45,215.35,-0.578569,-0.815633,0,0,6,...,173.69999,772.12599,0.00,772.12599,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01 00:15:00+02:00,4.1325,4.50,9.00,213.10,-0.546102,-0.837719,0,15,6,...,173.69999,772.12599,0.00,772.12599,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-01 00:30:00+02:00,3.4250,4.45,8.55,211.25,-0.518650,-0.854708,0,30,6,...,173.69999,772.12599,0.00,772.12599,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-01 00:45:00+02:00,2.7175,4.30,8.10,209.70,-0.495459,-0.868632,0,45,6,...,173.69999,772.12599,0.00,772.12599,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-01 01:00:00+02:00,2.0100,4.30,8.10,211.85,-0.527451,-0.849036,1,0,6,...,173.69999,772.12599,0.00,772.12599,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105211,2025-12-31 22:45:00+02:00,103.1100,-16.30,4.10,7.20,0.125333,0.992115,22,45,2,...,-1155.29000,-3158.88000,-2380.44,3158.88000,-2088.86,379.72,-2861.24,-585.91,801.42,1004.87
105212,2025-12-31 23:00:00+02:00,146.9000,-16.30,3.85,14.90,0.257076,0.966164,23,0,2,...,-1185.07000,-3028.08000,-2321.53,3028.08000,-2127.90,534.71,-2830.92,-454.95,718.92,1028.40
105213,2025-12-31 23:15:00+02:00,110.0000,-15.20,4.00,14.00,0.241922,0.970296,23,15,2,...,-1173.21000,-3096.36000,-2406.08,3096.36000,-2167.61,625.72,-2822.06,-353.70,667.14,1018.16
105214,2025-12-31 23:30:00+02:00,106.8200,-15.65,4.25,7.30,0.127021,0.991555,23,30,2,...,-1192.05000,-3002.62000,-2289.14,3002.62000,-2150.42,638.31,-2828.12,-339.77,695.48,1014.97


### Create train & test data set

Same chronological 80/20 split as V2.5 (`shuffle=False`) so the test set is
identical to the one used by the V2.5 notebook.

In [3]:
# drop the target and the datetime column from the feature set
X = df.drop(columns=['price', 'datetime'])
y = df['price']

In [4]:
from sklearn.model_selection import train_test_split

# 80/20 chronological split - no shuffle (time series!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [5]:
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('\nV3 feature columns (%d):' % X_train.shape[1])
print(X_train.columns.tolist())

# show which columns are new vs V2.5
v25 = pd.read_csv('../data/convertData/V2.5_15min_features.csv', nrows=0)
new_cols = [c for c in X_train.columns if c not in v25.columns]
print('\nNew V3 (grid) columns (%d):' % len(new_cols))
print(new_cols)

X_train shape: (84172, 62)
X_test shape : (21044, 62)

V3 feature columns (62):
['temp', 'wind_speed', 'wind_direction_deg', 'wind_dir_sin', 'wind_dir_cos', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'year', 'time_of_day', 'season', 'is_weekend', 'is_peak_hour', 'is_night_hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'week_of_year_sin', 'week_of_year_cos', 'is_holiday', 'is_non_working', 'price_lag_1', 'price_lag_2', 'price_lag_4', 'price_lag_8', 'price_lag_16', 'price_lag_32', 'price_lag_96', 'price_lag_672', 'price_rolling_mean_1h', 'price_rolling_std_1h', 'price_rolling_mean_6h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_min_24h', 'price_rolling_max_24h', 'price_rolling_mean_7d', 'temp_rolling_mean_1h', 'HDD', 'wind_power_proxy', 'temp_lag_4', 'temp_lag_96', 'fi_ee', 'fi_no', 'fi_se_north', 'fi_se_central', 'fi_se_total', 'fi_total_net', 'fi_se_abs', 'fi_total_net_lag_9

### TRAIN V3 XGBOOST REGRESSION MODEL

Same hyperparameters as V2.5 for a fair comparison.

In [9]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Fix: on Windows with GBK default encoding, sklearn's HTML estimator diagram
# (display='diagram') raises UnicodeDecodeError when it reads estimator.js.
# Force text-only display so the fitted model can render without the file read.
import sklearn
sklearn.set_config(display='text')

# same hyperparameters as V2.5
model_v3 = XGBRegressor(
    objective='reg:squarederror',
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=42)

model_v3.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
# evaluate on the test set
y_pred = model_v3.predict(X_test)

k = X_test.shape[1]   # number of features
n = len(X_test)       # number of samples

mse_score  = mean_squared_error(y_true=y_test, y_pred=y_pred)
rmse_score = np.sqrt(mse_score)
mae_score  = mean_absolute_error(y_true=y_test, y_pred=y_pred)
r2_score_val = r2_score(y_true=y_test, y_pred=y_pred)
adjusted_r2 = 1 - (1 - r2_score_val) * (n - 1) / (n - k - 1)

print('================ V3 XGBoost Evaluation ================')
print('Mean Absolute Error (MAE)            :', mae_score)
print('Mean Squared Error (MSE)             :', mse_score)
print('Root Mean Squared Error (RMSE)       :', rmse_score)
print('R2 Score (R2)                        :', r2_score_val)
print('Adjusted R2                           :', adjusted_r2)
print('========================================================')
print('\nV2.5 reference (recorded in modelV2.5.ipynb):')
print('  MAE 2.82 | RMSE 8.22 | R2 0.972')

================ V3 XGBoost Evaluation ================
Mean Absolute Error (MAE)            : 2.8467454006363524
Mean Squared Error (MSE)             : 70.01966429319445
Root Mean Squared Error (RMSE)       : 8.367775349111282
R2 Score (R2)                        : 0.9707814186736695
Adjusted R2                           : 0.9706950761712991

V2.5 reference (recorded in modelV2.5.ipynb):
  MAE 2.82 | RMSE 8.22 | R2 0.972


## Result — V2.5 vs V3 comparison

| Model | Features | MAE | RMSE | R² |
| ----- | -------- | --- | ---- | --- |
| V2.5  | 49       | **2.82** | **8.22** | **0.972** |
| V3    | 62 (49 + 13 grid) | 2.847 | 8.368 | 0.971 |

**Verdict: adding the 13 cross-border grid features did NOT improve the model —**
it is slightly WORSE on every metric (MAE +0.027, RMSE +0.148, R² −0.0012).

Same test set, same hyperparameters, same chronological split — only the feature
columns differ, so this is a fair controlled comparison.

This is an **honest negative result** (same methodology as the V2.5.1 risk-feature
experiment): a feature group that intuitively should help (more supply-side
information) does not help the model at the current default XGBoost settings.
Possible reasons: the model already captures the relevant signal through price
lags/rolling features; the grid flows are noisy at 15-min resolution; or the
default (untuned) XGBoost cannot exploit them.

**Decision: do NOT adopt V3.** No model is saved. Grid features remain available
in `V3_15min_features.csv` for future experiments (e.g. after Optuna tuning or as
part of the V4 nuclear comparison).

In [8]:
# ── Conclusion: negative result → model NOT saved ────────────────────────────
# V3 (V2.5 + 13 grid features) is slightly worse than V2.5 on the same test set.
# Following the project's controlled-experiment methodology (see V2.5.1), a feature
# group that does not improve the model is NOT adopted, so NO pkl is written.
# The grid features remain in V3_15min_features.csv for future experiments.

print('Final: V3 (with grid) vs V2.5')
print('  MAE  : 2.847 vs 2.82   -> slightly WORSE')
print('  RMSE : 8.368 vs 8.22   -> slightly WORSE')
print('  R2   : 0.971 vs 0.972  -> slightly WORSE')
print('Verdict: grid features did not help -> xgboost_v3.pkl NOT saved.')
print('(If you later want to revisit: set SAVE=True below and re-run.)')

SAVE = False  # keep False — negative result
if SAVE:
    import joblib
    from pathlib import Path
    save_dir = Path('../models/experiments')
    save_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump({
        'model': model_v3,
        'feature_cols': X_train.columns.tolist(),
        'step_min': 15,
    }, save_dir / 'xgboost_v3.pkl')
    print('Saved → models/experiments/xgboost_v3.pkl')

Final: V3 (with grid) vs V2.5
  MAE  : 2.847 vs 2.82   -> slightly WORSE
  RMSE : 8.368 vs 8.22   -> slightly WORSE
  R2   : 0.971 vs 0.972  -> slightly WORSE
Verdict: grid features did not help -> xgboost_v3.pkl NOT saved.
(If you later want to revisit: set SAVE=True below and re-run.)
